# 07 — Freeze, verify and package the release

**You will:** export the reviewed branch, validate the released PAR/TIM pairs,
preserve review/config provenance, write a content manifest and verify an extracted archive.
The output is a synthetic training release, clearly labelled as such.
PLEB's branch operations and fits produced the dataset; ordinary Python/Git below
package it. PLEB does not replace your scientific release approval.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
assert (ROOT/'review_approval.json').exists() and (ROOT/'model_approval.json').exists()
release = ROOT/'training-pta-release-v1'
assert not release.exists(), 'Use a new version/output name; do not overwrite a frozen release'
released_data = lab.snapshot(REPO,'release_model',release)
# Publish only model/TOA pairs that we actually fitted and checked. Preserve the
# other products as teaching examples, outside the released dataset directory.
for psr in PULSARS:
    for variant in ['legacy','new','combined']:
        for suffix in ['.par','_all.tim']:
            product = released_data/psr/f'{psr}_{variant}{suffix}'
            if product.exists():
                target = release/'provenance/variant_examples'/psr/product.name
                target.parent.mkdir(parents=True,exist_ok=True)
                shutil.move(product,target)
counts = lab.inventory(released_data)
display(counts)
expected_bad = len(json.loads((ROOT/'review_approval.json').read_text())['selected_bad'])
assert counts.physical.sum()==580 and counts.active.sum()==576-expected_bad
assert counts.extra_active_duplicates.sum()==0
lab.verify_hashes(ROOT/'raw',json.loads((ROOT/'raw_manifest.json').read_text())['sha256'])

Fit the **exported** pairs as a final smoke check. Files in an archive can differ from
the ones you inspected earlier. Missing INCLUDE files, mismatched models and clocks
must fail validation. The artificial station is defined by the supplied synthetic runtime;
it is not an observing site to use for real data.

In [ ]:
for psr in PULSARS:
    work = lab.simulation.project(ROOT/'export_validation'/psr)
    frame, stats = lab.simulation.fit_tempo2(work,released_data/psr/f'{psr}.par',
        released_data/psr/f'{psr}_all.tim','exported',epoch='58730',do_fit=False)
    assert len(frame)==int(counts.loc[counts.pulsar==psr,'active'].sum())
    assert np.isfinite(frame.residual_us).all()
    print(psr,'exported residual count',len(frame))

In [ ]:
provenance = release/'provenance'
provenance.mkdir(exist_ok=True)
for name in ['configs','review','logs']:
    shutil.copytree(ROOT/name,provenance/name)
# Preserve the exact original review input tables, including their relative
# paths, as well as the append-only override history and merged outputs.
qc_root = ROOT/'results/03_detect'
for relative,expected in json.loads((ROOT/'qc_input_hashes.json').read_text()).items():
    original = qc_root/relative
    assert hashlib.sha256(original.read_bytes()).hexdigest()==expected
    target = provenance/'qc_inputs'/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(original,target)
for name in ['software.json','raw_manifest.json','review_approval.json',
             'model_approval.json','final_fit_metrics.csv','qc_input_hashes.json']:
    shutil.copy2(ROOT/name,provenance/name)
# Save the exact mounted PLEB source used in this teaching run.
shutil.copytree('/pleb-source/pleb',provenance/'pleb',ignore=shutil.ignore_patterns('__pycache__','*.pyc'))
shutil.copytree(Path(os.environ['TEMPO2'])/'observatory',provenance/'synthetic-runtime/observatory')
shutil.copy2(Path(os.environ['TEMPO2'])/'clock/train2utc.clk',provenance/'synthetic-runtime/train2utc.clk')
commit = lab.command(['git','-C',REPO,'rev-parse','release_model']).strip()
(release/'README.txt').write_text('SYNTHETIC PTA TRAINING RELEASE. Not EPTA DR3.\n'
    'Canonical model: data/PSR/PSR.par; TOAs: data/PSR/PSR_all.tim.\n'
    'Unvalidated variant examples are under provenance/variant_examples, outside data/.\n'
    'Artificial station train requires the recorded runtime and the PLEB timing container.\n')
manifest = dict(synthetic=True,dataset_commit=commit,pulsars=PULSARS,
                counts=counts.to_dict('records'),files=lab.hashes(release))
(release/'manifest.json').write_text(json.dumps(manifest,indent=2))
lab.verify_hashes(release,{**manifest['files'],'manifest.json':hashlib.sha256((release/'manifest.json').read_bytes()).hexdigest()})
archive = ROOT/'training-pta-release-v1.tar.gz'
with tarfile.open(archive,'w:gz') as out:
    out.add(release,arcname=release.name)
checksum = hashlib.sha256(archive.read_bytes()).hexdigest()
Path(str(archive)+'.sha256').write_text(checksum+'  '+archive.name+'\n')
extracted = ROOT/'archive-verification'
assert not extracted.exists(), 'Choose a new verification directory'
with tarfile.open(archive) as saved:
    saved.extractall(extracted,filter='data')
copy = extracted/release.name
assert (copy/'manifest.json').read_bytes()==(release/'manifest.json').read_bytes()
lab.verify_hashes(copy,{**manifest['files'],'manifest.json':hashlib.sha256((copy/'manifest.json').read_bytes()).hexdigest()})
print('Synthetic release:',archive,'SHA256:',checksum)

**Your task:** alter a file in a separate extracted copy and demonstrate a hash failure.
Explain why matching hashes establish transfer integrity but not that all excluded TOAs
were scientifically bad. Your release needs both computational checks and review evidence.